In [1]:
# import packages
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd

In [2]:
# Read in files
years = [16, 17, 18, 19, 20, 21, 22]
df_list = []
base_path = "/Users/taoyiran/Documents/Github/datasci151spring2026/_final_project/2-FIFA/"

for y in years:
    file = base_path + "female_players_" + str(y) + ".csv"
    df = pd.read_csv(file)
    df_list.append(df)


# Merging procedure

First, from each csv file, only the relevant columns -- "sofifa_id", "overall", "potential", and "dob" -- were retained. Next, the variables that measure athelete performance ("overall" and "potential") were renamed by appending the corresponding year to prepare for later averaging. Before merging, the 2016 dataframe was set as the primary dataframe. To avoid duplication of the overlapping "dob" variable, the "dob" columns were dropped from the remaining dataframes. During merging, the dataframes are joined on the shared column "sofifa_id". After merging, the resulting dataframe only includes players who played consistently from 2016 to 2022.

In [3]:
# Keep only needed columns
cols = ["sofifa_id", "overall", "potential", "dob"]

for i in range(len(df_list)):
    df_list[i] = df_list[i][cols]

In [4]:
# Rename column names for later averaging
for i in range(len(df_list)):
    y = years[i]
    
    df_list[i] = df_list[i].rename(columns={"overall": "overall" + str(y),"potential": "potential" + str(y)})

In [5]:
# Merging

# Set 2016 as the primary dataframe
df_merge = df_list[0]

# Keep dob only in first dataframe to avoid overlapping
for i in range(1, len(df_list)):
    df_list[i] = df_list[i].drop(columns="dob")

# Merge
for i in range(1, len(df_list)):
    df_merge = pd.merge(left=df_merge,right=df_list[i],how="left",on="sofifa_id")


# Cleaning procedure

First, the "overall" and "potential" columns across all years were averaged to create two new columns "avg_overall" and "avg_potential", which represent the average performance of each player. Next, the "dob" column was inspected for missing values and format. After confirming that all "dob" entries are consistent in the "year-month-date" structure, each "dob" entry was split into three strings using hyphens. Then, the revised "dob" column was checked again to ensure that all entires follow the "year", "month", "date" structure. After that, the second component from each entry was extracted to create a new column called "month" to represent the birth month of each player. These months are grouped into four categories: "jan-mar", "apr-jun", "jul-sep", and "oct-dec", and stored in a new column called "month_group". Finally, the final dataframe only retained relevant variables: "sofifa_id", "avg_overall", "avg_potential", and "month_group".

In [6]:
# Compute average overall and potential

# Organize columns
overall_cols = []
potential_cols = []

for y in years:
    overall_cols.append("overall" + str(y))
    potential_cols.append("potential" + str(y))

# Create new columns
df_merge["avg_overall"] = df_merge[overall_cols].mean(axis=1)
df_merge["avg_potential"] = df_merge[potential_cols].mean(axis=1)

In [ ]:
# Inspect dob column

# Are there any missing values?
print(df_merge["dob"].isna().sum())

# Are all year, month, and date split by "-"?
print(df_merge["dob"].str.contains("-").mean())
# All year, month, and date split by "-".

0
1.0


In [8]:
# Extract month from dob

# Split dob into year, month, and date
df_merge["dob_parts"] = df_merge["dob"].str.split("-")

# Are all entires split in three parts?
lengths = df_merge["dob_parts"].apply(len)
print(pd.unique(lengths))
# All dob entries are in three parts.

# Extract month, the middle part
def get_month(x):
    return int(x[1])

df_merge["month"] = df_merge["dob_parts"].apply(get_month)

[3]


In [9]:
# Group months
month_group = []

for i in range(len(df_merge)):
    m = df_merge.iloc[i]["month"]
    
    if m <= 3:
        month_group.append("jan-mar")
    elif m <= 6:
        month_group.append("apr-jun")
    elif m <= 9:
        month_group.append("jul-sep")
    else:
        month_group.append("oct-dec")

df_merge["month_group"] = month_group

In [10]:
# Final clean
df_final = df_merge[["sofifa_id", "avg_overall", "avg_potential", "month_group"]]

display(df_final.head())

,sofifa_id,avg_overall,avg_potential,month_group
0,226324,89.5,89.5,jul-sep
1,226328,91.0,91.0,jul-sep
2,226334,90.0,90.0,apr-jun
3,226362,90.0,90.0,jan-mar
4,226373,89.0,89.0,apr-jun
